## Lab 5 — Workflow Design with LLMs (Prompt Templates & Output Parsers)

-----------------------------

### Exercise 1 — Customer Review to Support Ticket Pipeline

Build a 2-step pipeline. Step 1 extracts the core complaint, product/feature mentioned, and customer sentiment from a raw customer review as structured data. Step 2 generates a support ticket summary for the internal team, using only the structured data from Step 1, not the raw review text.

**Input:** `review_text`

**Output:** the structured complaint data from Step 1, and the support ticket summary from Step 2

**Step 1 — Template:** `PromptTemplate` &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: `complaint`, `product_or_feature`, `sentiment`)

**Step 2 — Template:** `ChatPromptTemplate` (system persona: "support ticket writer") &nbsp;|&nbsp; **Parser:** `StrOutputParser`

In [136]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from pydantic import BaseModel, Field

In [137]:
from dotenv import load_dotenv
load_dotenv()
import os 

In [138]:
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"]
)


In [139]:
input_temp_1=PromptTemplate.from_template(template="The customer review regarding the product is {customer_review}", input=["customer_review"])

In [140]:
input=input_temp_1.invoke({"customer_review":"The product is great, but the delivery was late."})

In [141]:
llm.invoke(input).content

"**Customer Review Summary**\n\n- **Positive Aspect:** The product itself is *great* – the reviewer is satisfied with its quality and performance.  \n- **Negative Aspect:** Delivery was *late* – the shipping time did not meet expectations.\n\n---\n\n### Sentiment Breakdown\n| Element | Sentiment | Notes |\n|---------|-----------|-------|\n| Product quality | Positive | “great” indicates strong satisfaction. |\n| Delivery time | Negative | “late” signals a service shortfall. |\n| Overall tone | Mixed (positive bias) | The praise for the product outweighs the delivery complaint, but the lateness is a notable issue. |\n\n---\n\n### Key Takeaways for the Business\n\n1. **Product Strength** – Keep maintaining the high quality; customers appreciate it.\n2. **Shipping Reliability** – Investigate the cause of the delay (warehouse, carrier, logistics).  \n3. **Communication** – Even if delays happen, proactive updates can mitigate frustration.\n4. **Post‑Delivery Follow‑Up** – A quick thank‑you

In [142]:
class CustomerReview(BaseModel):
    complaint: str = Field(description="The complaint of the customer")
    product: str = Field(description="The product the customer is reviewing")
    sentiment: str = Field(description="The sentiment of the customer's review")


In [143]:
pydantic_parser=PydanticOutputParser(pydantic_object=CustomerReview)

In [144]:
input_temp=ChatPromptTemplate([
    ("system", "You are an helpful assistant and a ticket support writer who takes customer complaints and writes support tickets."),
    ("user", "The customer review regarding the product is {customer_review}. Please extract the complaint, product, and sentiment from the review and provide it in a structured format."),
]).partial(JSON_FORMATTING=pydantic_parser.get_format_instructions())

In [145]:
response=llm.invoke(input)

In [146]:
str_parser= StrOutputParser().parse(response.content)
print(str_parser)

**Customer Review Summary**

- **Positive**: The product itself is well-received (“great”).
- **Negative**: Delivery was delayed (“delivery was late”).

**Sentiment**: Overall positive but with a notable complaint about shipping.

**Key Take‑aways**

| Aspect | Feedback | Recommendation |
|--------|----------|----------------|
| Product quality | Excellent | Continue maintaining high quality |
| Delivery time | Late | Review shipping partners, optimize logistics, consider expedited shipping options or clearer delivery windows |

**Suggested Response to the Customer**

> “Thank you for your feedback! We’re thrilled to hear you love the product. We apologize for the delay in delivery and are working to improve our shipping times. Please let us know if there’s anything else we can do to make your experience better.”

This addresses both the compliment and the issue, showing appreciation while acknowledging the delivery shortfall.


--------------------------------------


### Exercise 2 — Research Paper Abstract to Layperson Summary Pipeline

Build a 2-step pipeline. Step 1 extracts the research question, method, and key finding from a paper abstract as structured data. Step 2 generates a plain-language summary for a non-expert audience, using only the structured version from Step 1, not the original abstract.

**Input:** `abstract_text`

**Output:** the structured extraction from Step 1, and the layperson summary from Step 2

**Step 1 — Template:** `PromptTemplate` &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: `research_question`, `method`, `key_finding`)

**Step 2 — Template:** `ChatPromptTemplate` (system persona: "science communicator explaining to a general audience") &nbsp;|&nbsp; **Parser:** `StrOutputParser`



In [147]:
abstract_text = """This study investigates whether regular physical activity improves
memory in older adults. Researchers conducted a 12-month study involving
200 adults aged 60 and above. Participants were divided into an exercise
group and a control group. The results showed that participants who
exercised regularly demonstrated improved memory scores compared with
the control group."""

input_temp_2=PromptTemplate.from_template(template="The research paper abstract is {abstract_text}",input=["abstract_text"])

input=input_temp_2.invoke({"abstract_text":abstract_text})

llm.invoke(input).content

'**Concise Summary of the Abstract**\n\nThis 12‑month randomized study examined the impact of regular physical activity on memory in older adults.  \n- **Participants:** 200 adults aged 60+ were randomly assigned to an exercise group or a control group.  \n- **Intervention:** The exercise group engaged in a structured physical activity program throughout the study period.  \n- **Outcome:** Those who exercised consistently showed significantly better memory performance than the control group, indicating that regular physical activity can enhance memory in older adults.  \n\n**Key Takeaways**\n\n- **Population:** Community‑dwelling seniors (≥60 years).  \n- **Design:** Longitudinal, 12‑month duration.  \n- **Result:** Exercise → improved memory scores.  \n\nIf you’d like a deeper dive—such as methodological details, effect sizes, or potential limitations—just let me know!'

In [148]:
class ResearchAbstract(BaseModel):
    research_question: str = Field(description="The research question of the paper")
    method: str = Field(description="The research method used in the paper")
    key_finding: str = Field(description="The key finding of the research")

In [149]:
pydantic_parser=PydanticOutputParser(pydantic_object=ResearchAbstract)

In [150]:
input_temp=PromptTemplate.from_template(
    template="The research paper abstract is {abstract_text}.Please extract the research question, method, and key finding from the abstract.{JSON_FORMATTING}"
).partial(JSON_FORMATTING=pydantic_parser.get_format_instructions())

input=input_temp.invoke({"abstract_text":abstract_text})


In [151]:
response=llm.invoke(input)

In [152]:

structured_data=pydantic_parser.parse(response.content)

print(structured_data)

research_question='Does regular physical activity improve memory in older adults?' method='12‑month randomized controlled trial with 200 adults aged 60 and above divided into an exercise group and a control group.' key_finding='Participants who exercised regularly demonstrated improved memory scores compared with the control group.'


In [153]:
summary_temp=ChatPromptTemplate([
    ("system", "You are a science communicator explaining research to a general audience. Write a simple and clear summary using only the structured data provided."),
    ("user", """The research question is {research_question}.
    The method is {method}.
    The key finding is {key_finding}.

    Explain this research in simple language for a non-expert audience.""")
])

input=summary_temp.invoke({
    "research_question":structured_data.research_question,
    "method":structured_data.method,
    "key_finding":structured_data.key_finding
})

response=llm.invoke(input)

In [154]:
str_parser=StrOutputParser()

summary=str_parser.parse(response.content)

print(summary)

**What the study asked:**  
Does regular exercise help older adults keep their memory sharp?

**How the study was done:**  
A 12‑month experiment with 200 people aged 60 or older. Half of them did a regular exercise program, while the other half did not exercise (control group).

**What was found:**  
Those who exercised regularly scored better on memory tests than the people who didn’t exercise.


-----------

### Exercise 3 — Interview Transcript to Candidate Scorecard Pipeline

Build a 3-step pipeline. Step 1 extracts the candidate's answers to each interview question from a raw transcript. Step 2 evaluates each answer against a given set of skills, flagging any skill that wasn't clearly demonstrated. Step 3 formats the evaluation into a structured hiring scorecard with an overall recommendation.

**Input:** `transcript_text`, `skills_to_assess`

**Output:** the extracted answers from Step 1, the flagged skill evaluation from Step 2, and the final scorecard from Step 3

**Step 1 — Template:** `PromptTemplate` &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (field: list of `question` + `answer` pairs)

**Step 2 — Template:** `ChatPromptTemplate` (system persona: "technical interviewer evaluating skill evidence") &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: `skill`, `demonstrated: bool`, `evidence_or_gap`)

**Step 3 — Template:** `ChatPromptTemplate` (system persona: "hiring panel coordinator") &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: list of scorecard rows + `overall_recommendation`)

In [155]:
transcript_text = """
Interviewer: Explain your experience with Python.
Candidate: I have used Python for two years to automate tasks and clean datasets.

Interviewer: How do you work with databases?
Candidate: I have written SQL queries using joins and aggregate functions.

Interviewer: Describe your teamwork experience.
Candidate: I worked with three developers on a college project and reviewed their code.

Interviewer: Have you worked with cloud platforms?
Candidate: I have read about cloud computing but have not used it practically.
"""

skills_to_assess = ["Python", "SQL", "Teamwork", "Cloud Computing"]

In [156]:
class InterviewAnswer(BaseModel):
    question: str = Field(description="The interview question")
    answer: str = Field(description="The candidate's answer")

class InterviewTranscript(BaseModel):
    answers: list[InterviewAnswer] = Field(
        description="List of interview question and answer pairs"
    )

In [157]:
answer_parser=PydanticOutputParser(pydantic_object=InterviewTranscript)

In [158]:
input_temp=PromptTemplate.from_template(
    template="""Extract every interview question and its corresponding candidate answer
    from the following transcript.

    Transcript: {transcript_text}

    Return the result in the required JSON format.
    {JSON_FORMATTING}"""
).partial(JSON_FORMATTING=answer_parser.get_format_instructions())

input=input_temp.invoke({"transcript_text":transcript_text})

response=llm.invoke(input)

print(response.content)

extracted_answers=answer_parser.parse(response.content)

print(extracted_answers)

```json
{
  "answers": [
    {
      "question": "Explain your experience with Python.",
      "answer": "I have used Python for two years to automate tasks and clean datasets."
    },
    {
      "question": "How do you work with databases?",
      "answer": "I have written SQL queries using joins and aggregate functions."
    },
    {
      "question": "Describe your teamwork experience.",
      "answer": "I worked with three developers on a college project and reviewed their code."
    },
    {
      "question": "Have you worked with cloud platforms?",
      "answer": "I have read about cloud computing but have not used it practically."
    }
  ]
}
```
answers=[InterviewAnswer(question='Explain your experience with Python.', answer='I have used Python for two years to automate tasks and clean datasets.'), InterviewAnswer(question='How do you work with databases?', answer='I have written SQL queries using joins and aggregate functions.'), InterviewAnswer(question='Describe your teamw

In [159]:
class SkillEvaluation(BaseModel):
    skill: str = Field(description="The skill being assessed")
    demonstrated: bool = Field(
        description="Whether the candidate clearly demonstrated the skill"
    )
    evidence_or_gap: str = Field(
        description="Supporting evidence or explanation of missing evidence"
    )

class SkillEvaluations(BaseModel):
    evaluations: list[SkillEvaluation] = Field(
        description="List of evaluations for all requested skills"
    )

In [160]:
skill_parser=PydanticOutputParser(pydantic_object=SkillEvaluations)

In [161]:
evaluation_temp=ChatPromptTemplate([
    ("system", "You are a technical interviewer evaluating skill evidence. Evaluate every requested skill using only the extracted interview answers. Mark demonstrated as false if the skill is not clearly supported by evidence. Return valid JSON matching the required schema."),
    ("user", """Evaluate the following skills using the extracted answers.

    Extracted Answers: {extracted_answers}
    Skills to Assess: {skills_to_assess}

    Provide the skill, demonstrated status, and evidence or gap for every skill.

    {JSON_FORMATTING}""")
]).partial(JSON_FORMATTING=skill_parser.get_format_instructions())

input=evaluation_temp.invoke({
    "extracted_answers":extracted_answers.model_dump_json(),
    "skills_to_assess":", ".join(skills_to_assess)
})

response=llm.invoke(input)

print(response.content)

skill_evaluation=skill_parser.parse(response.content)

print(skill_evaluation)

```json
{
  "evaluations": [
    {
      "skill": "Python",
      "demonstrated": true,
      "evidence_or_gap": "Used Python for two years to automate tasks and clean datasets."
    },
    {
      "skill": "SQL",
      "demonstrated": true,
      "evidence_or_gap": "Written SQL queries using joins and aggregate functions."
    },
    {
      "skill": "Teamwork",
      "demonstrated": true,
      "evidence_or_gap": "Worked with three developers on a college project and reviewed their code."
    },
    {
      "skill": "Cloud Computing",
      "demonstrated": false,
      "evidence_or_gap": "Read about cloud computing but has not used it practically."
    }
  ]
}
```
evaluations=[SkillEvaluation(skill='Python', demonstrated=True, evidence_or_gap='Used Python for two years to automate tasks and clean datasets.'), SkillEvaluation(skill='SQL', demonstrated=True, evidence_or_gap='Written SQL queries using joins and aggregate functions.'), SkillEvaluation(skill='Teamwork', demonstrated=True,

In [162]:
class ScorecardRow(BaseModel):
    skill: str = Field(description="The skill assessed")
    demonstrated: bool = Field(description="Whether the skill was demonstrated")
    evidence_or_gap: str = Field(description="Evidence or identified gap")

class HiringScorecard(BaseModel):
    scorecard_rows: list[ScorecardRow] = Field(
        description="List of scorecard rows for all assessed skills"
    )
    overall_recommendation: str = Field(
        description="Overall hiring recommendation with evidence-based justification"
    )

In [163]:
scorecard_parser=PydanticOutputParser(pydantic_object=HiringScorecard)

In [164]:
scorecard_temp=ChatPromptTemplate([
    ("system", "You are a hiring panel coordinator. Prepare a structured hiring scorecard using only the provided skill evaluations. Preserve the evidence and gaps for each skill. Make an evidence-based overall recommendation without inventing qualifications."),
    ("user", """Prepare the final hiring scorecard from the following evaluations.

    Skill Evaluations: {skill_evaluation}

    Include a scorecard row for each skill and an overall recommendation
    justified by the available evidence.

    {JSON_FORMATTING}""")
]).partial(JSON_FORMATTING=scorecard_parser.get_format_instructions())

input=scorecard_temp.invoke({
    "skill_evaluation":skill_evaluation.model_dump_json()
})

response=llm.invoke(input)

print(response.content)

final_scorecard=scorecard_parser.parse(response.content)

print(final_scorecard)

```json
{
  "scorecard_rows": [
    {
      "skill": "Python",
      "demonstrated": true,
      "evidence_or_gap": "Used Python for two years to automate tasks and clean datasets."
    },
    {
      "skill": "SQL",
      "demonstrated": true,
      "evidence_or_gap": "Written SQL queries using joins and aggregate functions."
    },
    {
      "skill": "Teamwork",
      "demonstrated": true,
      "evidence_or_gap": "Worked with three developers on a college project and reviewed their code."
    },
    {
      "skill": "Cloud Computing",
      "demonstrated": false,
      "evidence_or_gap": "Read about cloud computing but has not used it practically."
    }
  ],
  "overall_recommendation": "Based on demonstrated proficiency in Python, SQL, and teamwork, the candidate is a strong fit for roles requiring data manipulation and collaborative development. The lack of practical cloud experience represents a gap that could be addressed through on-the-job training or a learning plan. Overall

-------------------

### Exercise 4 — Outdoor Event Plan to Weather-Adjusted Recommendation Pipeline

Build a 3-step pipeline. Step 1 extracts the event details (event type, planned date, location, indoor/outdoor preference) from a free-form event description. Step 2 calls a weather API for the extracted location and date, and normalizes the response into structured weather data. Step 3 generates a recommendation message for the event organizer (proceed as planned / reschedule / move indoors) that combines both the event details and the weather data.

**Input:** `event_description_text`

**Output:** the structured event details from Step 1, the structured weather data from Step 2, and the recommendation message from Step 3

**Step 1 — Template:** `PromptTemplate` &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: `event_type`, `event_date`, `location`, `indoor_outdoor_preference`)

**Step 2 — Template:** `PromptTemplate` (used to format the weather API call/query from Step 1's fields; the API response itself is normalized into structured output, not freely generated by the LLM) &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: `temperature`, `precipitation_chance`, `conditions`)

**Step 3 — Template:** `ChatPromptTemplate` (system persona: "event planning assistant") &nbsp;|&nbsp; **Parser:** `StrOutputParser`

In [165]:
event_description_text = """
We are organizing an outdoor birthday party at Pandit Lawns,
Pune, Maharashtra, India, on October 25, 2026.
We prefer to keep the event outdoors unless the weather makes
that difficult.
"""

input_temp=PromptTemplate.from_template(
    template="Extract the event details from this description: {event_description_text}",
    input=["event_description_text"]
)

input=input_temp.invoke({"event_description_text":event_description_text})


In [166]:

llm.invoke(input).content

'**Event Details**\n\n```json\n{\n  "event": "Birthday Party",\n  "location": {\n    "venue": "Pandit Lawns",\n    "city": "Pune",\n    "state": "Maharashtra",\n    "country": "India"\n  },\n  "date": "2026-10-25",\n  "time": "Not specified",\n  "venue_type": "Outdoor (unless weather conditions make it impractical)",\n  "weather_contingency": "If weather makes outdoor difficult, consider moving the event indoors."\n}\n```'

In [167]:
class EventDetails(BaseModel):
    event_type: str = Field(description="The type of event")
    event_date: str = Field(description="The planned event date in YYYY-MM-DD format")
    location: str = Field(description="The event location")
    indoor_outdoor_preference: str = Field(description="Whether the event is preferred indoors or outdoors")

In [168]:
event_parser=PydanticOutputParser(pydantic_object=EventDetails)

In [169]:
input_temp=PromptTemplate.from_template(
    template="""Extract the event type, planned date, location, and
    indoor/outdoor preference from the following description.

    Event Description: {event_description_text}

    Use YYYY-MM-DD for the date. Do not guess missing details.
    {JSON_FORMATTING}"""
).partial(JSON_FORMATTING=event_parser.get_format_instructions())

input=input_temp.invoke({
    "event_description_text":event_description_text
})

response=llm.invoke(input)

print(response.content)

event_details=event_parser.parse(response.content)

print(event_details)

```json
{
  "event_type": "birthday party",
  "event_date": "2026-10-25",
  "location": "Pandit Lawns, Pune, Maharashtra, India",
  "indoor_outdoor_preference": "outdoor"
}
```
event_type='birthday party' event_date='2026-10-25' location='Pandit Lawns, Pune, Maharashtra, India' indoor_outdoor_preference='outdoor'


In [170]:
class WeatherData(BaseModel):
    temperature: float = Field(description="Maximum forecast temperature in Celsius")
    precipitation_chance: float = Field(description="Maximum precipitation probability as a percentage")
    conditions: str = Field(description="Description of the forecast weather conditions")

In [171]:
weather_parser=PydanticOutputParser(pydantic_object=WeatherData)

In [172]:
from urllib.parse import quote

weather_query_temp=PromptTemplate.from_template(
    template="https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
)

geocoding_url=weather_query_temp.invoke({
    "location":quote(event_details.location)
}).text

print(geocoding_url)

https://geocoding-api.open-meteo.com/v1/search?name=Pandit%20Lawns%2C%20Pune%2C%20Maharashtra%2C%20India&count=1&language=en&format=json


In [173]:
import requests
import json

location = event_details.location

# Use the city extracted from the event description
geo_response = requests.get(
    "https://geocoding-api.open-meteo.com/v1/search",
    params={"name": "Pune", "count": 1},
    timeout=20
)
geo_response.raise_for_status()

geo_data = geo_response.json()

if not geo_data.get("results"):
    raise ValueError("Could not find Pune in the geocoding API")

latitude = geo_data["results"][0]["latitude"]
longitude = geo_data["results"][0]["longitude"]

weather_response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_max,precipitation_probability_max,weather_code",
        "start_date": event_details.event_date,
        "end_date": event_details.event_date,
        "timezone": "auto"
    },
    timeout=20
)
weather_response.raise_for_status()

daily = weather_response.json().get("daily", {})

if not daily.get("time"):
    raise ValueError(
        "Forecast unavailable for this date. "
        "Try a date within the available forecast range."
    )

normalized_weather = {
    "temperature": daily["temperature_2m_max"][0],
    "precipitation_chance": daily["precipitation_probability_max"][0],
    "conditions": str(daily["weather_code"][0])
}

weather_data = weather_parser.parse(json.dumps(normalized_weather))

print(weather_data)

temperature=32.9 precipitation_chance=8.0 conditions='3'


In [174]:
recommendation_temp=ChatPromptTemplate([
    ("system", "You are an event planning assistant. Recommend whether to proceed as planned, reschedule, or move indoors based on the event details and weather data. The conditions field contains a numeric weather code. Do not invent weather information."),
    ("user", """Prepare a recommendation for the event organizer.

    Event Type: {event_type}
    Event Date: {event_date}
    Location: {location}
    Indoor/Outdoor Preference: {indoor_outdoor_preference}

    Maximum Temperature: {temperature} °C
    Precipitation Chance: {precipitation_chance}%
    Weather Code: {conditions}

    Recommend whether to proceed as planned, reschedule, or move indoors.
    Explain your reasoning and give practical advice.""")
])

input=recommendation_temp.invoke({
    "event_type": event_details.event_type,
    "event_date": event_details.event_date,
    "location": event_details.location,
    "indoor_outdoor_preference": event_details.indoor_outdoor_preference,
    "temperature": weather_data.temperature,
    "precipitation_chance": weather_data.precipitation_chance,
    "conditions": weather_data.conditions
})

response=llm.invoke(input)

str_parser=StrOutputParser()

recommendation=str_parser.parse(response.content)

print(recommendation)

**Recommendation:**  
Proceed with the outdoor birthday party as planned.

**Why this is the best option**

| Factor | Details | Impact on decision |
|--------|---------|--------------------|
| **Maximum temperature** | 32.9 °C | Warm but not extreme. Most guests will be comfortable, especially if you provide shade and hydration. |
| **Precipitation chance** | 8 % | Very low likelihood of rain. Even if a brief shower occurs, the odds are small enough that the event can still run outdoors with minimal disruption. |
| **Weather code** | 3 | The numeric code alone does not tell us the exact conditions. Since we’re not given a description for code 3, we rely on the explicit temperature and precipitation data, which are favorable. |
| **Indoor/outdoor preference** | Outdoor | The plan is to host the party outside, which aligns with the favorable weather outlook. |

**Practical advice for a smooth outdoor event**

1. **Shade and shelter**  
   - Rent a lightweight canopy or pop‑up tent to pr

----------------------------

### Exercise 5 — Customer Support Chat Log to Escalation Report Pipeline

Build a 3-step pipeline. Step 1 normalizes a messy support chat log into a structured summary (issue, customer tone, resolution attempted). Step 2 checks the structured summary for missing information needed to escalate the issue (e.g. account ID, error code) and flags any gaps. Step 3 generates a prioritized escalation report for the support lead based on the structured, gap-checked version.

**Input:** `chat_log_text`

**Output:** the structured summary from Step 1, the flagged gaps from Step 2, and the escalation report from Step 3

**Step 1 — Template:** `PromptTemplate` &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: `issue`, `customer_tone`, `resolution_attempted`)

**Step 2 — Template:** `ChatPromptTemplate` (system persona: "escalation-readiness reviewer") &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: `missing_fields: list`, `is_ready_to_escalate: bool`)

**Step 3 — Template:** `ChatPromptTemplate` (system persona: "support operations lead") &nbsp;|&nbsp; **Parser:** `StrOutputParser`

In [175]:
chat_log_text = """
Customer: my account isn't working!! error says E104
Agent: Sorry about that. Have you tried resetting your password?
Customer: YES!! did that twice. still can't log in.
Agent: Please clear your browser cache and try again.
Customer: already tried that too. this is getting really frustrating.
Agent: I understand. We may need to escalate this issue to our technical team.
Customer: please do. I need access urgently.
"""

input_temp=PromptTemplate.from_template(
    template="The customer support chat log is {chat_log_text}",
    input=["chat_log_text"]
)

input=input_temp.invoke({"chat_log_text":chat_log_text})

llm.invoke(input).content

'**Escalation Ticket – Customer #E104**\n\n| Field | Detail |\n|-------|--------|\n| **Customer** | “my account isn’t working!! error says E104” |\n| **Issue** | Unable to log in after password reset; error code E104 persists. |\n| **Steps Taken** | 1. Password reset (twice) <br>2. Browser cache cleared |\n| **Current Status** | Account remains inaccessible; customer urgently needs access. |\n| **Priority** | High (urgent business impact) |\n| **Suggested Next Steps** | • Verify that the account is not locked or flagged for security. <br>• Check for any recent changes to authentication services or IP restrictions. <br>• Re‑enable the account manually if needed. <br>• Provide a temporary login link or password reset if standard methods fail. |\n| **Contact** | Customer (direct line or email) – *[Insert contact info]* |\n| **Assigned Team** | Technical Support – Authentication & Security |\n| **Due Date** | Within 2 hours of receipt to meet the customer’s urgency. |\n\n---\n\n**Message t

In [176]:
class SupportSummary(BaseModel):
    issue: str = Field(description="The main customer issue")
    customer_tone: str = Field(description="The customer's tone or emotional state")
    resolution_attempted: str = Field(description="Troubleshooting steps already attempted")

In [177]:
summary_parser=PydanticOutputParser(pydantic_object=SupportSummary)

In [178]:
input_temp=PromptTemplate.from_template(
    template="""Summarize the customer support chat log by extracting
    the issue, customer tone, and resolution attempts.

    Chat Log: {chat_log_text}

    {JSON_FORMATTING}"""
).partial(JSON_FORMATTING=summary_parser.get_format_instructions())

input=input_temp.invoke({"chat_log_text":chat_log_text})

response=llm.invoke(input)

print(response.content)

structured_summary=summary_parser.parse(response.content)

print(structured_summary)

```json
{
  "issue": "Unable to log into account, receiving error E104",
  "customer_tone": "Frustrated and urgent",
  "resolution_attempted": "Password reset attempted twice; browser cache cleared"
}
```
issue='Unable to log into account, receiving error E104' customer_tone='Frustrated and urgent' resolution_attempted='Password reset attempted twice; browser cache cleared'


In [179]:
class EscalationReadiness(BaseModel):
    missing_fields: list[str] = Field(
        description="Required escalation information that is missing, such as account ID or error code"
    )
    is_ready_to_escalate: bool = Field(
        description="Whether sufficient information is available to escalate the issue"
    )

In [180]:
readiness_parser=PydanticOutputParser(pydantic_object=EscalationReadiness)

In [181]:
readiness_temp=ChatPromptTemplate([
    ("system", "You are an escalation-readiness reviewer. Check the structured support summary for missing information needed for escalation, including account ID, error code, and relevant troubleshooting details. Do not invent missing information."),
    ("user", """Review the following structured support summary.

    Issue: {issue}
    Customer Tone: {customer_tone}
    Resolution Attempted: {resolution_attempted}

    Identify missing information required for escalation.
    Set is_ready_to_escalate to true only if enough information is available
    to proceed with escalation. Otherwise, set it to false.

    {JSON_FORMATTING}""")
]).partial(JSON_FORMATTING=readiness_parser.get_format_instructions())

input=readiness_temp.invoke({
    "issue":structured_summary.issue,
    "customer_tone":structured_summary.customer_tone,
    "resolution_attempted":structured_summary.resolution_attempted
})

response=llm.invoke(input)

print(response.content)

readiness_result=readiness_parser.parse(response.content)

print(readiness_result)

```json
{
  "missing_fields": ["account ID"],
  "is_ready_to_escalate": false
}
```
missing_fields=['account ID'] is_ready_to_escalate=False


In [182]:
report_temp=ChatPromptTemplate([
    ("system", "You are a support operations lead. Generate a prioritized escalation report using only the structured support summary and escalation-readiness review. Highlight urgency, troubleshooting already attempted, missing information, and the recommended next action."),
    ("user", """Generate an escalation report using the following information.

    Issue: {issue}
    Customer Tone: {customer_tone}
    Resolution Attempted: {resolution_attempted}

    Missing Fields: {missing_fields}
    Ready to Escalate: {is_ready_to_escalate}

    Include priority, issue summary, troubleshooting history, missing
    information, and recommended next action.""")
])

input=report_temp.invoke({
    "issue":structured_summary.issue,
    "customer_tone":structured_summary.customer_tone,
    "resolution_attempted":structured_summary.resolution_attempted,
    "missing_fields":", ".join(readiness_result.missing_fields) or "None",
    "is_ready_to_escalate":readiness_result.is_ready_to_escalate
})

response=llm.invoke(input)

str_parser=StrOutputParser()

escalation_report=str_parser.parse(response.content)

print(escalation_report)

**Escalation Report – Support Summary**

| Item | Details |
|------|---------|
| **Issue** | Unable to log into account, receiving error **E104** |
| **Customer Tone** | Frustrated & urgent |
| **Priority** | **High** – Immediate resolution required to restore access for a frustrated customer |
| **Issue Summary** | Customer cannot log in and receives error code E104. No other authentication failures have been reported. |
| **Troubleshooting History** | 1. Password reset attempted twice.<br>2. Browser cache cleared.<br>3. No success; error persists. |
| **Missing Information** | **Account ID** (required to locate the user record and verify account status). |
| **Escalation‑Readiness** | **Not ready** – additional data needed before moving to higher tiers. |

---

### Recommended Next Action

1. **Collect the missing Account ID**  
   * Ask the customer to provide the account ID or the email address linked to the account.  
   * Verify the account exists in the system and that it is not

------------------------

## Exercise 6 — Workout Goal to Weekly Plan Pipeline

Build a 2-step pipeline. Step 1 extracts fitness goal details (goal type, current fitness level, days available per week, equipment access) from a free-form fitness goal description. Step 2 generates a structured weekly workout plan scaled to the extracted availability, using only the extracted details from Step 1, not the original description.

**Input:** `fitness_goal_text`, `days_per_week`

**Output:** the extracted goal details from Step 1, and the weekly workout plan from Step 2

**Step 1 — Template:** `PromptTemplate` &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (fields: `goal_type`, `fitness_level`, `days_available`, `equipment_access`)

**Step 2 — Template:** `ChatPromptTemplate` (system persona: "certified personal trainer") &nbsp;|&nbsp; **Parser:** `PydanticOutputParser` (field: list of day-by-day plan entries)

In [183]:
fitness_goal_text = """
I want to improve my overall fitness and build strength.
I am a beginner and can work out four days per week.
I have a pair of dumbbells, a yoga mat, and access to a gym occasionally.
"""

days_per_week = 4

input_temp=PromptTemplate.from_template(
    template="The fitness goal description is {fitness_goal_text}. Days available per week: {days_per_week}",
    input=["fitness_goal_text", "days_per_week"]
)

input=input_temp.invoke({
    "fitness_goal_text":fitness_goal_text,
    "days_per_week":days_per_week
})

llm.invoke(input).content

'## 4‑Day Beginner Strength & Fitness Plan  \n*(Designed for a beginner who has a pair of dumbbells, a yoga mat, and occasional gym access. The plan focuses on overall fitness, building strength, and establishing a sustainable routine.)*\n\n---\n\n### 📅 Weekly Schedule\n\n| Day | Workout | Focus | Notes |\n|-----|---------|-------|-------|\n| **Mon** | **Upper‑Body Strength** | Push & Pull | 4‑5 min dynamic warm‑up |\n| **Tue** | **Lower‑Body Strength** | Squat & Hinge | 4‑5 min dynamic warm‑up |\n| **Wed** | **Rest / Active Recovery** | Light yoga / mobility | 15–20 min |\n| **Thu** | **Upper‑Body Strength** | Push & Pull + Core | 4‑5 min dynamic warm‑up |\n| **Fri** | **Lower‑Body Strength** | Squat & Hinge + Core | 4‑5 min dynamic warm‑up |\n| **Sat‑Sun** | **Rest** | |  |\n\n> **Tip**: If you have gym access, swap a dumbbell exercise for a machine or barbell for variety.\n\n---\n\n## 1️⃣ Warm‑Up (All workout days)\n\n| Exercise | Duration / Reps | Purpose |\n|----------|-----------

In [184]:
class FitnessGoal(BaseModel):
    goal_type: str = Field(description="The main fitness goal")
    fitness_level: str = Field(description="The current fitness level")
    days_available: int = Field(description="Number of days available per week")
    equipment_access: str = Field(description="Available workout equipment and gym access")

In [185]:
fitness_parser=PydanticOutputParser(pydantic_object=FitnessGoal)

In [186]:
input_temp=PromptTemplate.from_template(
    template="""Extract the fitness goal type, current fitness level,
    days available per week, and equipment access from the following description.

    Fitness Goal: {fitness_goal_text}
    Days Available Per Week: {days_per_week}

    {JSON_FORMATTING}"""
).partial(JSON_FORMATTING=fitness_parser.get_format_instructions())

input=input_temp.invoke({
    "fitness_goal_text":fitness_goal_text,
    "days_per_week":days_per_week
})

response=llm.invoke(input)

print(response.content)

fitness_details=fitness_parser.parse(response.content)

print(fitness_details)

```json
{
  "goal_type": "Improve overall fitness and build strength",
  "fitness_level": "beginner",
  "days_available": 4,
  "equipment_access": "pair of dumbbells, a yoga mat, and access to a gym occasionally"
}
```
goal_type='Improve overall fitness and build strength' fitness_level='beginner' days_available=4 equipment_access='pair of dumbbells, a yoga mat, and access to a gym occasionally'


In [187]:
class WorkoutDay(BaseModel):
    day: str = Field(description="Day of the week")
    focus: str = Field(description="Workout focus for the day")
    exercises: list[str] = Field(description="Exercises with sets and repetitions or duration")
    duration: str = Field(description="Estimated workout duration")

class WeeklyWorkoutPlan(BaseModel):
    weekly_plan: list[WorkoutDay] = Field(
        description="Day-by-day workout plan for the week"
    )

In [188]:
workout_parser=PydanticOutputParser(pydantic_object=WeeklyWorkoutPlan)

In [189]:
workout_temp=ChatPromptTemplate([
    ("system", "You are a certified personal trainer. Create a realistic weekly workout plan using only the extracted fitness goal details provided. Scale the plan to the person's available days, fitness level, and equipment."),
    ("user", """Generate a structured weekly workout plan using these extracted details.

    Goal Type: {goal_type}
    Fitness Level: {fitness_level}
    Days Available: {days_available}
    Equipment Access: {equipment_access}

    {JSON_FORMATTING}""")
]).partial(JSON_FORMATTING=workout_parser.get_format_instructions())

input=workout_temp.invoke({
    "goal_type":fitness_details.goal_type,
    "fitness_level":fitness_details.fitness_level,
    "days_available":fitness_details.days_available,
    "equipment_access":fitness_details.equipment_access
})

response=llm.invoke(input)

print(response.content)

weekly_plan=workout_parser.parse(response.content)

print(weekly_plan)

```json
{
  "weekly_plan": [
    {
      "day": "Monday",
      "focus": "Full Body Strength",
      "exercises": [
        "Goblet Squat: 3 sets x 12 reps",
        "Dumbbell Bench Press: 3 sets x 10 reps",
        "Bent Over Row: 3 sets x 12 reps",
        "Dumbbell Deadlift: 3 sets x 10 reps",
        "Bodyweight Lunge: 3 sets x 12 reps per leg",
        "Plank: 3 sets x 30 seconds"
      ],
      "duration": "45 minutes"
    },
    {
      "day": "Tuesday",
      "focus": "Upper Body & Core",
      "exercises": [
        "Dumbbell Shoulder Press: 3 sets x 10 reps",
        "Push‑ups: 3 sets x 12 reps",
        "Bent Over Reverse Fly: 3 sets x 12 reps",
        "Triceps Dips (bench): 3 sets x 10 reps",
        "Biceps Curl: 3 sets x 12 reps",
        "Russian Twist: 3 sets x 20 reps",
        "Side Plank: 3 sets x 20 seconds per side"
      ],
      "duration": "40 minutes"
    },
    {
      "day": "Thursday",
      "focus": "Lower Body & Cardio",
      "exercises": [
        "Dumb

-------------------------------------------------